In [38]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /home/plont/Documents/Praktikum_Big_Data/tugas
!hdfs dfs -put -f transaksi_tugas5.csv /home/plont/Documents/Praktikum_Big_Data/tugas
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /home/plont/Documents/Praktikum_Big_Data/tugas/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas")

!hdfs dfs -get /home/plont/Documents/Praktikum_Big_Data/tugas/transaksi_tugas5.csv transaksi.csv

df = pd.read_csv("transaksi.csv")
print(df)

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /home/plont/Documents/Praktikum_Big_Data/tugas/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas
get: `transaksi.csv': File exists
    order_id                kategori        kota  unit_terjual  harga_satuan
0      TRX-0       Makanan & Minuman   Purworejo             8         75000
1      TRX-1              Elektronik        Solo             9         75000
2      TRX-2  Kesehatan & Kecantikan        Solo             6        100000
3      TRX-3                 Fashion  Yogyakarta             6        100000
4      TRX-4              Elektronik  Yogyakarta             2         75000
..       ...                     ...         ...           ...           ...
495  TRX-495            Rumah Tangga    Semarang             8         75000
496  TRX-496                 Fashion  Yogyakarta             8         50000
497  TRX-497              Elektronik  Yog

A. Join & Perbandingan Target (bobot 25%)

Ringkas total pendapatan per kota dari df_transaksi, lalu join dengan df_target. Tambahkan kolom pencapaian_persen. Urutkan hasil dari pencapaian tertinggi.

In [50]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, round
import pandas as pd

spark = SparkSession.builder.appName("Tugas5").getOrCreate()

target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(target_cabang))

path_hdfs = "/home/plont/Documents/Praktikum_Big_Data/tugas/transaksi_tugas5.csv"
df_transaksi = spark.read.csv(path_hdfs, header=True, inferSchema=True)

df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))
df_pendapatan_kota = df_transaksi.groupBy("kota").agg(sum("pendapatan").alias("total_pendapatan"))

df_hasil = df_pendapatan_kota.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", round((col("total_pendapatan") / col("target_bulanan")) * 100, 2)) \
    .orderBy(col("pencapaian_persen").desc())

df_hasil.show()

+----------+----------------+--------------+----------+-----------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang|pencapaian_persen|
+----------+----------------+--------------+----------+-----------------+
| Purworejo|        45650000|      30000000|     Fitri|           152.17|
|      Solo|        33475000|      40000000|      Bayu|            83.69|
|Yogyakarta|        47275000|      60000000|      Joko|            78.79|
|  Magelang|        31650000|      45000000|      Rani|            70.33|
|  Semarang|        38175000|      55000000|      Sari|            69.41|
+----------+----------------+--------------+----------+-----------------+



B. Window Function — Kategori Terlaris per Kota (bobot 25%)

Menggunakan window function, tentukan kategori dengan pendapatan tertinggi di setiap kota (top-1 saja, gunakan row_number()).

In [48]:
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when, expr, col, sum

spark = SparkSession.builder \
    .appName("transaksi_tugas5") \
    .getOrCreate()

!hdfs dfs -get /home/plont/Documents/Praktikum_Big_Data/tugas/transaksi_tugas5.csv transaksi.csv

df = pd.read_csv("transaksi.csv")

dtf = spark.createDataFrame(pd.DataFrame(df))

df_pendapatan = dtf.groupBy("kota", "kategori") \
                  .agg(expr("sum(unit_terjual * harga_satuan)").alias("total_pendapatan"))

windowSpec = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

df_tertinggi = df_pendapatan.withColumn("rn", row_number().over(windowSpec)) \
                            .filter(col("rn") == 1) \
                            .drop("rn")

df_tertinggi.show()

get: `transaksi.csv': File exists
+----------+--------------------+----------------+
|      kota|            kategori|total_pendapatan|
+----------+--------------------+----------------+
|  Magelang|Kesehatan & Kecan...|         7275000|
| Purworejo|Kesehatan & Kecan...|        10075000|
|  Semarang|        Rumah Tangga|        11125000|
|      Solo|Kesehatan & Kecan...|         8425000|
|Yogyakarta|             Fashion|        13325000|
+----------+--------------------+----------------+



C. Spark SQL (bobot 25%)

Daftarkan df_transaksi dan df_target sebagai temporary view, lalu tulis satu kueri SQL (bukan DataFrame API) yang menampilkan: kota, pic_cabang, dan jumlah transaksi (COUNT) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.

In [60]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

spark.sql("""
    SELECT 
        transaksi.kota, 
        target.pic_cabang, 
        COUNT(transaksi.order_id) AS jumlah_transaksi
    FROM transaksi 
    JOIN target ON transaksi.kota = target.kota
    GROUP BY transaksi.kota, target.pic_cabang
    ORDER BY jumlah_transaksi DESC
""").show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



D. Kesimpulan (bobot 25%)

Tulis pada markdown cell (minimal 100 kata): berdasarkan hasil bagian A dan B, cabang mana yang berkinerja paling baik dan cabang mana yang paling perlu perhatian manajemen? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.

Menurut data yang dihasilkan dari code bagian A dan code bagian B, cabang yang memiliki kinerja paling baik jika dihitung dari target bulanan adalah cabang yang berada di kota purworejo dengan Fitri sebagai penanggung jawab telah pencapaian mencapai 152.17 persen, jauh melampaui cabang lain. Cabang dibawahnya hanya mampu mencapai 83.69 persen saja, jarak antar keduanya bahkan mencapai 68.48 persen, hampir mendekati target dari cabang terendah.

Menurut data yang dihasilkan dari code bagian A dan code bagian B, cabang yang masih perlu diatur atau di manajemen lagi adalah cabang yang berada di kota semarang dengan Sari sebagai penanggung jawab. Target yang berhasil dipenuhi cabang ini hanya sekitar 69.41 persen, dinilai cukup jauh dari target yang telah ditetapkan yaitu cabang ini masih perlu memenuhi 16825000.